In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

# Reading from bronze tables

In [0]:

df = spark.table("workspace.bronze.crm_cust_info")

# Data transformation

In [0]:
df = df.select([
    trim(col(field.name)).alias(field.name) if isinstance(field.dataType, StringType)
    else col(field.name)
    for field in df.schema.fields
])



In [0]:

%sql
SELECT * FROM workspace.bronze.crm_cust_info
WHERE cst_firstname != trim(cst_firstname)

In [0]:
#give descriptive names for columns
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

#Renaming the columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Write into silver table

In [0]:
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver.crm_customers")

In [0]:
%sql
select * from workspace.silver.crm_customers